```markdown
# Simulação de Eventos Discretos de Fila de Atendimento

Este notebook apresenta o desenvolvimento de um simulador de eventos discretos utilizando a biblioteca **SimPy** para modelar o fluxo de atendimento com base nos dados reais coletados em três pranchetas (CSVs):

1. **Observador de Chegadas**: Modelagem da taxa de chegada de clientes (processo de Poisson local com taxa $\lambda$).
2. **Observador de Atendimento**: Modelagem dos tempos de serviço do caixa (distribuição exponencial/empírica para obter $\mu$).
3. **Rastreador de Espera**: Dados reais utilizados para validação das métricas de desempenho.

### Estrutura do Código:
- **Tratamento e Análise Exploratória**: Conversão de tipos de dados, cálculo de métricas empíricas e preparação de distribuições.
- **Motor SimPy**: Processo estocástico que replica o comportamento da fila com parâmetros configuráveis.
- **Validação e Comparação**: Geração de um painel de comparação direta entre os dados simulados e os dados reais.
- **Visualizações com Plotly Express**: Gráficos dinâmicos que representam o comportamento temporal do sistema e a acurácia do modelo.
```

```markdown
## Explicação Detalhada do Pipeline do Código

Abaixo está a descrição funcional de cada bloco desenvolvido neste projeto para garantir o correto entendimento e modularidade do simulador:

### 1. Carga, Higienização e Tratamento de Dados (Célula `691c5080`)
* **Objetivo**: Extrair os dados brutos e prepará-los para a simulação, resolvendo inconsistências de cabeçalho e ruídos das planilhas reais.
* **Busca Dinâmica de Cabeçalhos**: Filtra e elimina notas manuscritas e metadados nas primeiras linhas das planilhas, localizando automaticamente onde começam as colunas de dados reais (ex: procurando palavras-chave como `'Horário'`, `'Amostra'` ou `'Chegada'`).
* **Estimativa Estatística**:
  * Calcula a taxa de chegada de clientes por minuto ($\lambda$) baseada nas pessoas que de fato chegaram ao longo de 60 minutos (Prancheta 1).
  * Extrai os tempos individuais de serviço do caixa, convertendo segundos para minutos para unificar a base de tempo (Prancheta 2).
  * Calcula os tempos reais de espera e permanência no sistema a partir dos registros de entrada e saída (Prancheta 3) para fins de comparação posterior.

### 2. Motor de Simulação SimPy (Célula `27e0e622`)
* **Objetivo**: Executar a simulação dinâmica de eventos discretos baseada no comportamento empírico.
* **Classe `SistemaAtendimento`**: Modela a infraestrutura física (o caixa como um `simpy.Resource` com capacidade unitária) e dispara um processo paralelo em segundo plano para monitorar ativamente o tamanho da fila a cada 6 segundos simulados (`0.1 min`).
* **Processo do Cliente (`cliente_process`)**: Controla a jornada estocástica de cada indivíduo. O cliente chega ao sistema, aguarda na fila até que o recurso (caixa) esteja livre, consome um tempo de serviço sorteado diretamente a partir da distribuição empírica real (amostragem da Prancheta 2) e desocupa o canal.
* **Gerador de Clientes (`gerador_clientes`)**: Cria novas entidades seguindo uma distribuição exponencial de tempo entre chegadas baseada em $\lambda$ (Processo de Poisson).

### 3. Validação do Modelo (Célula `f80c3678`)
* **Objetivo**: Consolidar os dados simulados e compará-los matematicamente contra as métricas reais registradas na Prancheta 3.
* **Cálculo de Utilização ($U$)**: Mede a proporção do tempo total em que o caixa esteve ocupado em relação ao período da simulação (60 minutos).
* **Geração da Tabela Comparativa**: Apresenta de forma tabulada as diferenças de médias agregadas de tempo de fila, de atendimento e de utilização para validação cruzada rápida no console.

### 4. Visualizações Interativas (Célula `1cf2f137`)
* **Objetivo**: Traduzir graficamente o comportamento estocástico do sistema de filas em gráficos elegantes e interativos utilizando `plotly.express`.
* **Gráfico de Linhas (Fila ao longo do Tempo)**: Permite observar a transição temporal e identificar em quais minutos exatos ocorreram picos de lotação.
* **Histograma Comparativo**: Apresenta de forma sobreposta as curvas de distribuição empírica real vs. os resultados teóricos/estocásticos da simulação.
* **Comparação de Médias**: Gráfico de barras simples para facilitar a visualização do erro ou aderência das principais métricas do modelo.
* **Boxplot de Espera por Bloco**: Analisa a variabilidade e a dispersão dos tempos de espera simulados de forma fatiada a cada 5 minutos do expediente do caixa.
```

In [14]:
# Instalação da biblioteca de simulação SimPy caso não esteja instalada
!pip install -q simpy

In [15]:
import pandas as pd
import numpy as np
import simpy
import random
from datetime import datetime
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Definir semente aleatória para reprodutibilidade
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ==========================================
# 1. CARGA E TRATAMENTO DOS DADOS
# ==========================================

try:
    # Carregar dados brutos
    df_chegadas_raw = pd.read_csv('/content/Prancheta 1 - Observador de Chegadas - Página1.csv')
    df_atendimento_raw = pd.read_csv('/content/Prancheta 2 - Observador de Atendimento - Página1.csv')
    df_espera_raw = pd.read_csv('/content/Prancheta 3 - Rastreador de Espera - Página1.csv')

    # Encontrar as linhas onde começam os dados reais baseado em colunas-chave
    idx_chegadas = df_chegadas_raw.apply(lambda r: r.astype(str).str.contains('Horário').any(), axis=1).idxmax()
    df_chegadas = pd.read_csv('/content/Prancheta 1 - Observador de Chegadas - Página1.csv', skiprows=int(idx_chegadas) + 1)
    df_chegadas.columns = [c.strip() for c in df_chegadas.columns]

    idx_atendimento = df_atendimento_raw.apply(lambda r: r.astype(str).str.contains('Amostra').any(), axis=1).idxmax()
    df_atendimento = pd.read_csv('/content/Prancheta 2 - Observador de Atendimento - Página1.csv', skiprows=int(idx_atendimento) + 1)
    df_atendimento.columns = [c.strip() for c in df_atendimento.columns]

    idx_espera = df_espera_raw.apply(lambda r: r.astype(str).str.contains('Chegada').any(), axis=1).idxmax()
    df_espera = pd.read_csv('/content/Prancheta 3 - Rastreador de Espera - Página1.csv', skiprows=int(idx_espera) + 1)
    df_espera.columns = [c.strip() for c in df_espera.columns]

    print("Arquivos carregados e higienizados com sucesso!")
except Exception as e:
    print(f"Erro no tratamento inicial de arquivos: {e}")

# Tratamento Prancheta 1 (Chegadas)
total_pessoas = pd.to_numeric(df_chegadas['Total de Pessoas'], errors='coerce').dropna().sum()
tempo_total_minutos = 60
taxa_chegada_minuto = total_pessoas / tempo_total_minutos
intervalo_medio_chegadas = 1.0 / taxa_chegada_minuto

# Tratamento Prancheta 2 (Atendimento)
tempos_atendimento_seg = pd.to_numeric(df_atendimento['Tempo do Atendimento (segundos)'], errors='coerce').dropna().values
tempos_atendimento_min = tempos_atendimento_seg / 60.0
tempo_atendimento_medio_min = np.mean(tempos_atendimento_min)
taxa_servico_minuto = 1.0 / tempo_atendimento_medio_min

# Identificar colunas na Prancheta 3 usando busca parcial robusta
col_tempo_fila = [c for c in df_espera.columns if 'Tempo' in c or 'Fila' in c or 'Espera' in c][-1]
col_chegada = [c for c in df_espera.columns if 'Chegada' in c or 'Hora Chegada' in c][0]
col_saida = [c for c in df_espera.columns if 'Saída' in c or 'Saida' in c][0]

# Tratamento Prancheta 3 (Validação Real)
def parse_time_to_minutes(time_str):
    if pd.isna(time_str):
        return np.nan
    try:
        parts = list(map(int, str(time_str).strip().split(':')))
        if len(parts) == 3:
            return parts[0] * 60 + parts[1] + parts[2] / 60.0
        elif len(parts) == 2:
            return parts[0] + parts[1] / 60.0
    except:
        return np.nan

def parse_hour_relative(time_str):
    if pd.isna(time_str):
        return np.nan
    try:
        t = datetime.strptime(str(time_str).strip(), "%H:%M:%S")
        minutos_absolutos = t.hour * 60 + t.minute + t.second / 60.0
        return max(0.0, minutos_absolutos - 1050.0)  # Referência: 17:30 (1050 min)
    except:
        try:
            t = datetime.strptime(str(time_str).strip(), "%H:%M")
            minutos_absolutos = t.hour * 60 + t.minute
            return max(0.0, minutos_absolutos - 1050.0)
        except:
            return np.nan

if df_espera[col_tempo_fila].dtype == object:
    df_espera['Tempo Total na Fila (Minutos)'] = df_espera[col_tempo_fila].apply(parse_time_to_minutes)
else:
    df_espera['Tempo Total na Fila (Minutos)'] = pd.to_numeric(df_espera[col_tempo_fila], errors='coerce') / 60.0

tempo_medio_fila_real = df_espera['Tempo Total na Fila (Minutos)'].dropna().mean()

df_espera['Minutos_Chegada'] = df_espera[col_chegada].apply(parse_hour_relative)
df_espera['Minutos_Saida'] = df_espera[col_saida].apply(parse_hour_relative)
df_espera['Tempo_Sistema_Real'] = df_espera['Minutos_Saida'] - df_espera['Minutos_Chegada']
tempo_medio_sistema_real = df_espera['Tempo_Sistema_Real'].dropna().mean()

print(f"\nParâmetros do Modelo Calculados:")
print(f" - Taxa média de Chegada (lambda): {taxa_chegada_minuto:.4f} clientes/min")
print(f" - Tempo médio de Atendimento: {tempo_atendimento_medio_min*60:.2f} segundos")
print(f" - Tempo Médio Real na Fila (Prancheta 3): {tempo_medio_fila_real*60:.2f} segundos")

Arquivos carregados e higienizados com sucesso!

Parâmetros do Modelo Calculados:
 - Taxa média de Chegada (lambda): 2.2667 clientes/min
 - Tempo médio de Atendimento: 7.16 segundos
 - Tempo Médio Real na Fila (Prancheta 3): 288.85 segundos


In [16]:
# Exibindo os DataFrames limpos e tratados
print("=== Prancheta 1: Observador de Chegadas (Tratada) ===")
display(df_chegadas.head())

print("\n=== Prancheta 2: Observador de Atendimento (Tratada) ===")
display(df_atendimento.head())

print("\n=== Prancheta 3: Rastreador de Espera (Tratada com Tempos Convertidos) ===")
display(df_espera[[col_chegada, col_saida, col_tempo_fila, 'Tempo Total na Fila (Minutos)', 'Tempo_Sistema_Real']].head())

=== Prancheta 1: Observador de Chegadas (Tratada) ===


,Horário,Bloco de Tempo,Minutos,Total de Pessoas
0,17:30,Bloco 1,00 - 05 min,7
1,17:35,Bloco 2,05 - 10 min,1
2,17:40,Bloco 3,10 - 15 min,16
3,17:45,Bloco 4,15 - 20 min,5
4,17:50,Bloco 5,20 - 25 min,5



=== Prancheta 2: Observador de Atendimento (Tratada) ===


,Amostra,Tempo do Atendimento (segundos),Observações / Imprevistos
0,1,26.0,NaN
1,2,2.0,NaN
2,3,2.0,NaN
3,4,2.0,NaN
4,5,3.0,NaN



=== Prancheta 3: Rastreador de Espera (Tratada com Tempos Convertidos) ===


/usr/local/lib/python3.13/dist-packages/google/colab/_dataframe_summarizer.py:88: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.

/usr/local/lib/python3.13/dist-packages/google/colab/_dataframe_summarizer.py:88: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.

/usr/local/lib/python3.13/dist-packages/google/colab/_dataframe_summarizer.py:88: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



,Hora Chegada na Fila (hh:mm:ss),Hora Saída do Caixa (hh:mm:ss),Tempo Total na Fila (Min e Seg),Tempo Total na Fila (Minutos),Tempo_Sistema_Real
0,NaN,NaN,NaN,NaN,NaN
1,17:30:00,17:45:50,15:50,15.833333,15.833333
2,17:31:22,17:47:27,16:05,16.083333,16.083333
3,17:32:47,17:47:55,15:08,15.133333,15.133333
4,17:35:49,17:48:12,12:23,12.383333,12.383333


In [17]:
# ==========================================
# 2. MOTOR DE SIMULAÇÃO COM TAXA DINÂMICA (SimPy)
# ==========================================

class SistemaAtendimento:
    def __init__(self, env, num_caixas):
        self.env = env
        self.servidor = simpy.Resource(env, capacity=num_caixas)
        self.historico_fila = []
        self.historico_sistema = []
        self.tempos_registro = []

        # Para monitorar o tamanho da fila continuamente
        self.env.process(self.monitorar_tamanho_fila())

    def monitorar_tamanho_fila(self):
        while True:
            tempo_atual = self.env.now
            tam_fila = len(self.servidor.queue)
            em_atendimento = len(self.servidor.users)
            self.historico_fila.append(tam_fila)
            self.historico_sistema.append(tam_fila + em_atendimento)
            self.tempos_registro.append(tempo_atual)
            yield self.env.timeout(0.1) # Monitoramento a cada 6 segundos simulados

def cliente_process(env, nome, sistema, log_resultados):
    chegada = env.now

    # Solicitar atendimento no caixa
    with sistema.servidor.request() as request:
        yield request
        inicio_atendimento = env.now

        # Amostrar o tempo de atendimento da distribuição empírica real (Prancheta 2)
        tempo_servico = random.choice(tempos_atendimento_min)
        yield env.timeout(tempo_servico)

        fim_atendimento = env.now
        espera_fila = inicio_atendimento - chegada
        tempo_sistema = fim_atendimento - chegada

        log_resultados.append({
            'Cliente': nome,
            'Chegada': chegada,
            'Inicio_Atendimento': inicio_atendimento,
            'Fim_Atendimento': fim_atendimento,
            'Espera_Fila': espera_fila,
            'Tempo_Servico': tempo_servico,
            'Tempo_Sistema': tempo_sistema
        })

# Criar dicionário de taxas de chegada reais para cada intervalo de 5 minutos
taxas_por_bloco = {}
for idx, row in df_chegadas.iterrows():
    # Cada bloco i representa o intervalo de tempo [(i)*5, (i+1)*5]
    taxa_bloco = row['Total de Pessoas'] / 5.0 # clientes por minuto naquele bloco
    taxas_por_bloco[idx] = taxa_bloco

def gerador_clientes(env, sistema, taxas_blocos, log_resultados):
    id_cliente = 0
    while True:
        tempo_atual = env.now
        bloco_atual = int(tempo_atual // 5)

        # Garantir limite máximo de índice para blocos mapeados
        bloco_atual = min(bloco_atual, len(taxas_blocos) - 1)
        taxa_atual = taxas_blocos.get(bloco_atual, 0.0)

        if taxa_atual == 0.0:
            # Evitar divisão por zero caso haja blocos sem nenhum cliente previsto
            yield env.timeout(1.0)
            continue

        # Intervalo de tempo estocástico baseado na taxa instantânea
        intervalo = random.expovariate(taxa_atual)
        yield env.timeout(intervalo)
        id_cliente += 1
        env.process(cliente_process(env, f"Cliente_{id_cliente}", sistema, log_resultados))

# Resetar e executar a simulação dinâmica usando semente fixa
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

env = simpy.Environment()
sistema = SistemaAtendimento(env, num_caixas=1)
resultados_simulacao = []

env.process(gerador_clientes(env, sistema, taxas_por_bloco, resultados_simulacao))
env.run(until=tempo_total_minutos)

# Criar DataFrame dos resultados simulados
df_simulado = pd.DataFrame(resultados_simulacao)
display(df_simulado.head())

,Cliente,Chegada,Inicio_Atendimento,Fim_Atendimento,Espera_Fila,Tempo_Servico,Tempo_Sistema
0,Cliente_1,0.728614,0.728614,0.795281,0.000000,0.066667,0.066667
1,Cliente_2,0.746707,0.795281,0.845281,0.048575,0.050000,0.098575
2,Cliente_3,0.947345,0.947345,0.997345,0.000000,0.050000,0.050000
3,Cliente_4,1.899911,1.899911,1.966578,0.000000,0.066667,0.066667
4,Cliente_5,2.863944,2.863944,2.913944,0.000000,0.050000,0.050000


In [18]:
# ==========================================
# 3. VALIDAÇÃO DO MODELO
# ==========================================

# Métricas Simuladas
tempo_medio_espera_sim = df_simulado['Espera_Fila'].mean()
tempo_medio_sistema_sim = df_simulado['Tempo_Sistema'].mean()
tempo_medio_servico_sim = df_simulado['Tempo_Servico'].mean()

# Utilização do Caixa U (Tempo total ocupado / Tempo total de simulação)
tempo_total_ocupado = df_simulado['Tempo_Servico'].sum()
utilizacao_caixa_sim = min(1.0, tempo_total_ocupado / tempo_total_minutos)

# Utilização Real Estimada (Fórmula de Little: U = lambda * E[S])
utilizacao_real_estimada = min(1.0, taxa_chegada_minuto * (tempos_atendimento_min.mean()))

# Tabela Comparativa
tabela_comparativa = pd.DataFrame({
    'Métrica': [
        'Tempo Médio na Fila (segundos)',
        'Tempo Médio no Sistema (segundos)',
        'Tempo Médio de Serviço (segundos)',
        'Utilização do Servidor (U)'
    ],
    'Dados Reais (Prancheta 3)': [
        tempo_medio_fila_real * 60.0,
        tempo_medio_sistema_real * 60.0,
        tempos_atendimento_seg.mean(),
        f"{utilizacao_real_estimada * 100:.2f}%"
    ],
    'Simulação (SimPy)': [
        tempo_medio_espera_sim * 60.0,
        tempo_medio_sistema_sim * 60.0,
        tempo_medio_servico_sim * 60.0,
        f"{utilizacao_caixa_sim * 100:.2f}%"
    ]
})

display(tabela_comparativa)

,Métrica,Dados Reais (Prancheta 3),Simulação (SimPy)
0,Tempo Médio na Fila (segundos),288.851852,5.748984
1,Tempo Médio no Sistema (segundos),288.851852,13.019254
2,Tempo Médio de Serviço (segundos),7.156863,7.27027
3,Utilização do Servidor (U),27.04%,22.42%


In [19]:
# ==========================================
# 4. VISUALIZAÇÕES COM PLOTLY EXPRESS
# ==========================================

# a) Gráfico de Linhas/Área: Tamanho da fila N(t) ao longo do tempo
df_timeline = pd.DataFrame({
    'Tempo (min)': sistema.tempos_registro,
    'Pessoas na Fila': sistema.historico_fila,
    'Pessoas no Sistema': sistema.historico_sistema
})

fig_timeline = px.line(
    df_timeline,
    x='Tempo (min)',
    y=['Pessoas na Fila', 'Pessoas no Sistema'],
    title='Evolução da Fila e Ocupação do Sistema ao Longo do Tempo (Simulação)',
    labels={'value': 'Número de Clientes', 'variable': 'Métrica'},
    template='plotly_white'
)
fig_timeline.show()

# b) Histogramas de Comparação do Tempo Total no Sistema (Reais vs. Simulados)
df_espera_tmp = pd.DataFrame({'Tempo_Segundos': df_espera['Tempo_Sistema_Real'] * 60.0, 'Origem': 'Dados Reais'})
df_sim_tmp = pd.DataFrame({'Tempo_Segundos': df_simulado['Tempo_Sistema'] * 60.0, 'Origem': 'Simulação'})
df_distribuicoes = pd.concat([df_espera_tmp, df_sim_tmp]).dropna()

fig_hist = px.histogram(
    df_distribuicoes,
    x='Tempo_Segundos',
    color='Origem',
    barmode='overlay',
    nbins=30,
    title='Comparação da Distribuição do Tempo Total no Sistema',
    labels={'Tempo_Segundos': 'Tempo Total (segundos)'},
    template='plotly_white',
    marginal='box'
)
fig_hist.show()

# c) Gráfico de Barras: Comparativo de Métricas Gerais
df_bar = pd.DataFrame({
    'Métrica': ['Tempo Espera', 'Tempo Serviço', 'Tempo Sistema', 'Tempo Espera', 'Tempo Serviço', 'Tempo Sistema'],
    'Tempo (Segundos)': [
        tempo_medio_fila_real * 60.0,
        tempos_atendimento_seg.mean(),
        tempo_medio_sistema_real * 60.0,
        tempo_medio_espera_sim * 60.0,
        tempo_medio_servico_sim * 60.0,
        tempo_medio_sistema_sim * 60.0
    ],
    'Grupo': ['Real', 'Real', 'Real', 'Simulado', 'Simulado', 'Simulado']
})

fig_bar = px.bar(
    df_bar,
    x='Métrica',
    y='Tempo (Segundos)',
    color='Grupo',
    barmode='group',
    title='Comparativo Direto de Tempos Médios: Real vs. Simulação',
    template='plotly_white'
)
fig_bar.show()

# d) Boxplot: Distribuição dos tempos de espera simulados por bloco de tempo de 5 minutos
df_simulado['Bloco_Tempo'] = (df_simulado['Chegada'] // 5) * 5
df_simulado['Bloco_Tempo_Str'] = df_simulado['Bloco_Tempo'].apply(lambda x: f"{int(x)}-{int(x+5)} min")

fig_box = px.box(
    df_simulado,
    x='Bloco_Tempo_Str',
    y='Espera_Fila',
    title='Distribuição dos Tempos de Espera na Fila por Bloco de Chegada (Simulação)',
    labels={'Espera_Fila': 'Espera na Fila (Minutos)', 'Bloco_Tempo_Str': 'Bloco de Chegada (Minutos)'},
    template='plotly_white',
    category_orders={'Bloco_Tempo_Str': sorted(df_simulado['Bloco_Tempo_Str'].unique(), key=lambda x: int(x.split('-')[0]))}
)
fig_box.show()

In [20]:
# Diagnóstico de escala e valores originais da Prancheta 3 (Rastreador de Espera)
print("=== DIAGNÓSTICO DOS TEMPOS REAIS DE ESPERA (PRANCHETA 3) ===\n")

# Exibir os valores brutos originais do CSV para entender a formatação
df_espera_raw_headers = pd.read_csv('/content/Prancheta 3 - Rastreador de Espera - Página1.csv')
idx_esp = df_espera_raw_headers.apply(lambda r: r.astype(str).str.contains('Chegada').any(), axis=1).idxmax()
df_diagnostico = pd.read_csv('/content/Prancheta 3 - Rastreador de Espera - Página1.csv', skiprows=int(idx_esp) + 1)

col_tempo_original = [c for c in df_diagnostico.columns if 'Tempo' in c or 'Fila' in c or 'Espera' in c][-1]
print(f"Coluna identificada: '{col_tempo_original}'")
print("Primeiros 15 registros brutos coletados da fila:")
display(df_diagnostico[[col_tempo_original]].head(15))

# Verificar se a conversão do parser gerou valores excessivamente inflados
print("\nValores convertidos pelo código atual (em minutos):")
display(df_espera[[col_tempo_fila, 'Tempo Total na Fila (Minutos)']].head(15))

=== DIAGNÓSTICO DOS TEMPOS REAIS DE ESPERA (PRANCHETA 3) ===

Coluna identificada: 'Tempo Total na Fila (Min e Seg)'
Primeiros 15 registros brutos coletados da fila:


/usr/local/lib/python3.13/dist-packages/google/colab/_dataframe_summarizer.py:88: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



,Tempo Total na Fila (Min e Seg)
0,NaN
1,15:50
2,16:05
3,15:08
4,12:23
5,00:11
6,00:06
7,00:32
8,00:37
9,NaN



Valores convertidos pelo código atual (em minutos):


/usr/local/lib/python3.13/dist-packages/google/colab/_dataframe_summarizer.py:88: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



,Tempo Total na Fila (Min e Seg),Tempo Total na Fila (Minutos)
0,NaN,NaN
1,15:50,15.833333
2,16:05,16.083333
3,15:08,15.133333
4,12:23,12.383333
5,00:11,0.183333
6,00:06,0.100000
7,00:32,0.533333
8,00:37,0.616667
9,NaN,NaN


```markdown
### Ajuste do Modelo: Adicionando Atrito Humano e Operacional

Como observado, o ato de aproximar o QR Code é rápido (~7s), mas gargalos humanos (procurar o celular, falha de leitura, travamentos da catraca) adicionam atrito ao tempo de serviço real na fila.

Abaixo, vamos rodar uma versão da simulação onde cada atendimento tem uma chance de sofrer um pequeno atraso de preparação ou falha de leitura, para ver como as métricas se comportam.
```

In [21]:
# Resetar semente
random.seed(RANDOM_SEED)

# Nova simulação com atraso operacional (Atrito Humano)
resultados_atrito = []

def cliente_atrito_process(env, nome, sistema, log_resultados):
    chegada = env.now
    with sistema.servidor.request() as request:
        yield request
        inicio_atendimento = env.now

        # Tempo base de leitura de QR Code
        tempo_servico = random.choice(tempos_atendimento_min)

        # Simular Atrito Humano: 25% de chance de problemas (procurar celular, falha de leitura, etc.)
        # que somam entre 15 a 45 segundos extras (0.25 a 0.75 min)
        if random.random() < 0.25:
            tempo_servico += random.uniform(15/60, 45/60)

        yield env.timeout(tempo_servico)
        fim_atendimento = env.now

        log_resultados.append({
            'Cliente': nome,
            'Chegada': chegada,
            'Espera_Fila': inicio_atendimento - chegada,
            'Tempo_Servico': tempo_servico,
            'Tempo_Sistema': fim_atendimento - chegada
        })

env_atrito = simpy.Environment()
sistema_atrito = SistemaAtendimento(env_atrito, num_caixas=1)

# Registrar gerador adaptado
def gerador_atrito(env, sistema, taxas_blocos, log_resultados):
    id_cliente = 0
    while True:
        tempo_atual = env.now
        bloco_atual = min(int(tempo_atual // 5), len(taxas_blocos) - 1)
        taxa_atual = taxas_blocos.get(bloco_atual, 0.0)
        if taxa_atual == 0.0:
            yield env.timeout(1.0)
            continue
        yield env.timeout(random.expovariate(taxa_atual))
        id_cliente += 1
        env.process(cliente_atrito_process(env, f"Cliente_{id_cliente}", sistema, log_resultados))

env_atrito.process(gerador_atrito(env_atrito, sistema_atrito, taxas_por_bloco, resultados_atrito))
env_atrito.run(until=tempo_total_minutos)

df_atrito = pd.DataFrame(resultados_atrito)

print(f"=== SIMULAÇÃO AJUSTADA COM ATRITO OPERACIONAL ===")
print(f"Tempo Médio na Fila com Atrito: {df_atrito['Espera_Fila'].mean() * 60:.2f} segundos")
print(f"Tempo Médio Real Registrado: {tempo_medio_fila_real * 60:.2f} segundos")

=== SIMULAÇÃO AJUSTADA COM ATRITO OPERACIONAL ===
Tempo Médio na Fila com Atrito: 28.84 segundos
Tempo Médio Real Registrado: 288.85 segundos


```markdown
### 5. Simulação com Bloqueio de Fluxo (Tandem Queue com Capacidade de Triagem Limita)

Vamos modelar conceitualmente o efeito da triagem de comida de capacidade limitada pós-QR Code:
1. Área de Triagem comporta no máximo **5 pessoas** (`capacidade_triagem`).
2. O tempo para preparar e pegar a comida é estimado em média em **1.2 minutos** (distribuição exponencial).
3. Se a triagem encher, o leitor de QR Code fica bloqueado (bloqueio por retroalimentação ou *blocking*).
```

In [22]:
# Resetar semente
random.seed(RANDOM_SEED)

resultados_bloqueio = []

class SistemaComBloqueio:
    def __init__(self, env, cap_triagem=5):
        self.env = env
        self.leitor_qr = simpy.Resource(env, capacity=1)
        self.espaco_triagem = simpy.Resource(env, capacity=cap_triagem) # Buffer limite da comida

def cliente_bloqueio_process(env, nome, sistema, log_resultados):
    chegada = env.now

    # 1. Fila para o QR Code
    with sistema.leitor_qr.request() as req_qr:
        yield req_qr
        inicio_qr = env.now

        # Tempo rápido de leitura do QR Code
        tempo_leitura = random.choice(tempos_atendimento_min)
        yield env.timeout(tempo_leitura)

        # 2. Tentar entrar no espaço de triagem física
        # Se estiver cheio, o cliente fica "travando" o leitor de QR Code até liberar espaço
        with sistema.espaco_triagem.request() as req_triagem:
            yield req_triagem

            # Agora ele liberou o leitor de QR Code para o próximo
            fim_leitura_e_liberacao = env.now
            espera_fila = inicio_qr - chegada

            # Tempo preparando/recebendo a comida na triagem (estimativa de 1.2 min)
            tempo_preparo = random.expovariate(1.0 / 1.2)
            yield env.timeout(tempo_preparo)

        fim_sistema = env.now

        log_resultados.append({
            'Cliente': nome,
            'Chegada': chegada,
            'Espera_Fila': espera_fila,
            'Tempo_Bloqueado_No_Leitor': (fim_leitura_e_liberacao - inicio_qr) - tempo_leitura,
            'Tempo_Sistema': fim_sistema - chegada
        })

env_bloq = simpy.Environment()
sistema_bloq = SistemaComBloqueio(env_bloq, cap_triagem=5)

def gerador_bloqueio(env, sistema, taxas_blocos, log_resultados):
    id_cliente = 0
    while True:
        tempo_atual = env.now
        bloco_atual = min(int(tempo_atual // 5), len(taxas_blocos) - 1)
        taxa_atual = taxas_blocos.get(bloco_atual, 0.0)
        if taxa_atual == 0.0:
            yield env.timeout(1.0)
            continue
        yield env.timeout(random.expovariate(taxa_atual))
        id_cliente += 1
        env.process(cliente_bloqueio_process(env, f"Cliente_{id_cliente}", sistema, log_resultados))

env_bloq.process(gerador_bloqueio(env_bloq, sistema_bloq, taxas_por_bloco, resultados_bloqueio))
env_bloq.run(until=tempo_total_minutos)

df_bloq = pd.DataFrame(resultados_bloqueio)

print(f"=== SIMULAÇÃO COM BLOQUEIO DE TRIAGEM ===")
print(f"Tempo Médio de Fila Simulado com Bloqueio: {df_bloq['Espera_Fila'].mean() * 60:.2f} segundos")
print(f"Tempo Médio Real na Fila (Prancheta 3): {tempo_medio_fila_real * 60:.2f} segundos")

=== SIMULAÇÃO COM BLOQUEIO DE TRIAGEM ===
Tempo Médio de Fila Simulado com Bloqueio: 565.08 segundos
Tempo Médio Real na Fila (Prancheta 3): 288.85 segundos
